In [3]:
import urllib.request
import zipfile
import os 
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"

data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

In [4]:
def download_data_and_unzip(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download.")

    with urllib.request.urlopen(url) as response: 
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    with zipfile.ZipFile(zip_path,"r") as zip_ref:
        zip_ref.extractall(extracted_path)
    
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"File downloaded and saved as {data_file_path}")

In [6]:
download_data_and_unzip(url=url, zip_path=zip_path, extracted_path=extracted_path,
                        data_file_path=data_file_path)

File downloaded and saved as sms_spam_collection/SMSSpamCollection.tsv


In [10]:
import pandas as pd
df = pd.read_csv(data_file_path, sep="\t",header=None, names=["Label", "Text"])

In [12]:
print(df['Label'].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


In [ ]:
num_spam = df[df["Label"] == "spam"].shape[0]
ham_subset = df[df["Label"] == "ham"].sample(num_spam,random_state=123)
balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])                        
balanced_df["Label"] = balanced_df["Label"].map({"spam":0, "ham": 1})

In [ ]:
from sklearn.model_selection import train_test_split
def random_split(df, train_frac, validation_frac):

    df = df.sample(frac=1, random_state=123).reset_index(drop=True)
    train_ind = int(len(df) * train_frac)
    validation_ind = train_ind + int(len(df)*validation_frac)
    train_df = df[:train_ind]
    validation_df = df[train_ind:validation_ind]
    test_df = df[validation_ind:]
    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(df, train_frac=0.7, validation_frac=0.1)
#train_df, test_df = train_test_split(df, train_size=0.8, test_size=0.2)
#train_df, validation_df = train_test_split(train_df, train_size=0.85, test_size=0.15)

if not os.path.exists("../spam_dataset/"):
    os.mkdir("../spam_dataset")

train_df.to_csv('../spam_dataset/train.csv')
test_df.to_csv('../spam_dataset/test.csv')
validation_df.to_csv('../spam_dataset/validation.csv')
